# Med-DDPM v2 — Kaggle Training Notebook
Conditional Diffusion for Brain MRI Synthesis (single-channel FLAIR)

Based on: Dorjsembe et al. 2024
Adapted for Kaggle environment

## 1. Kaggle Setup & GPU Check

In [ ]:
import os
import torch

# Kaggle paths
KAGGLE_INPUT = "/kaggle/input"
KAGGLE_WORKING = "/kaggle/working"

# GPU check
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} ({gpu.total_memory/1e9:.1f} GB)")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("⚠️  No GPU detected! Enable GPU in Kaggle settings (Accelerator → GPU T4 x2 or P100)")

## 2. Install Dependencies

In [ ]:
!pip install -q tifffile scikit-image pyyaml albumentations tqdm matplotlib
print("Dependencies installed")

## 3. Clone Repo & Find Dataset

In [ ]:
import os, glob

os.chdir(KAGGLE_WORKING)

# Clone repo
if not os.path.exists('Mask-to-MRI'):
    !git clone https://github.com/AmineAitLaamim/Mask-to-MRI.git
    print("✅ Repository cloned.")
else:
    %cd Mask-to-MRI
    !git pull
    print("✅ Repository updated.")

os.chdir('Mask-to-MRI')

# Find the LGG dataset in Kaggle input
candidates = glob.glob(f"{KAGGLE_INPUT}/*/lgg-mri-segmentation")
if candidates:
    RAW_DIR = candidates[0]
    print(f"Found dataset: {RAW_DIR}")
else:
    print("⚠️  Dataset not found. Add it to Kaggle as a Dataset.")
    print(f"Available in {KAGGLE_INPUT}:")
    for d in os.listdir(KAGGLE_INPUT):
        print(f"  {d}")
    RAW_DIR = None

## 4. Create Output Directories

In [ ]:
OUTPUTS_DIR = f"{KAGGLE_WORKING}/outputs_v2"
for d in ["checkpoints", "samples", "metrics", "synthetic"]:
    os.makedirs(f"{OUTPUTS_DIR}/{d}", exist_ok=True)
    print(f"Created {OUTPUTS_DIR}/{d}")

## 5. Import Modules & Load Config

In [ ]:
# ── Force clear any cached src modules ──
import sys
for _mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[_mod]

sys.path.insert(0, '/kaggle/working/Mask-to-MRI')

from src.med_ddpm_v2 import ConditionalDDPM, CONFIG
from src.med_ddpm_v2.train import train
from src.dataset import build_flair_dataloaders

# Override config for Kaggle paths
if RAW_DIR:
    CONFIG["raw_dir"] = RAW_DIR
CONFIG["timesteps"] = 1000
CONFIG["checkpoint_dir"] = f"{OUTPUTS_DIR}/checkpoints"
CONFIG["samples_dir"] = f"{OUTPUTS_DIR}/samples"
CONFIG["metrics_dir"] = f"{OUTPUTS_DIR}/metrics"
CONFIG["synthetic_dir"] = f"{OUTPUTS_DIR}/synthetic"

print("Config loaded:")
print(f"  raw_dir: {CONFIG['raw_dir']}")
print(f"  Timesteps: {CONFIG['timesteps']}")
print(f"  Epochs: {CONFIG['epochs']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  in_channels: {CONFIG['in_channels']} (flair + mask)")
print(f"  out_channels: {CONFIG['out_channels']} (flair only)")

## 6. Device Setup & Seeds

In [ ]:
import random
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

seed = CONFIG["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
print(f"Random seed fixed: {seed}")

## 7. Build FLAIR Dataloaders

In [ ]:
loaders = build_flair_dataloaders(
    CONFIG["raw_dir"],
    batch_size=CONFIG["batch_size"],
    num_workers=2,
    tumor_ratio=CONFIG["tumor_ratio"],
    seed=CONFIG["seed"],
)

print(f"DataLoaders created:")
print(f"  Train: {len(loaders['train'].dataset)} samples")
print(f"  Val:   {len(loaders['val'].dataset)} samples")
print(f"  Batches/epoch (train): {len(loaders['train'])}")

## 8. Verify Batch Shape

In [ ]:
mask_batch, flair_batch = next(iter(loaders["train"]))
print(f"Batch shape: mask={mask_batch.shape}, flair={flair_batch.shape}")
print(f"FLAIR mean: {flair_batch[:,0].mean():.3f}, std: {flair_batch[:,0].std():.3f}")
print(f"Mask range: [{mask_batch.min():.2f}, {mask_batch.max():.2f}]")
print(f"FLAIR range: [{flair_batch.min():.2f}, {flair_batch.max():.2f}]")

# Must be (B, 1, 256, 256) — NOT (B, 3, 256, 256)
assert flair_batch.shape[1] == 1, f"Expected 1 channel, got {flair_batch.shape[1]}"
print("✅ Single-channel FLAIR verified")

## 9. Create Model

In [ ]:
model = ConditionalDDPM(CONFIG).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model created: {n_params:,} parameters ({n_params/1e6:.2f}M)")

## 10. Pre-Training Sanity Check

In [ ]:
print("=" * 60)
print("PRE-TRAINING SANITY CHECK")
print("=" * 60)

# Quick forward pass
mask_d = mask_batch[:2].to(device)
flair_d = flair_batch[:2].to(device)
loss = model(flair_d, mask_d)
print(f"Forward pass OK — loss={loss.item():.4f}")

# Quick sample
import time
start = time.time()
with torch.no_grad():
    fake = model.sample(mask_d[:1], ddim_steps=10)
elapsed = time.time() - start
print(f"Sample (10 DDIM steps): shape={fake.shape}, std={fake.std():.3f}")
print(f"10 steps took: {elapsed:.1f}s")
print(f"Full 250 steps estimated: {elapsed * 25:.0f}s")

print("=" * 60)
print("✅ SANITY CHECK COMPLETE — ready to train")
print("=" * 60)

## 11. Auto-Resume from Checkpoint

In [ ]:
import glob
from pathlib import Path

checkpoints = glob.glob(f"{CONFIG['checkpoint_dir']}/checkpoint_v2_epoch_*.pt")

if checkpoints:
    resume_from = max(checkpoints, key=lambda p: int(Path(p).stem.split('_')[-1]))
    print(f"Found checkpoint: {resume_from}")
    print("   Training will auto-resume from that epoch")
else:
    print("No checkpoint found — starting from scratch.")
    resume_from = None

## 12. Start Training

In [ ]:
history = train(
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    model=model,
    config=CONFIG,
    device=device,
    resume_from=resume_from,
)

## 13. Plot Training Loss

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = f"{CONFIG['metrics_dir']}/v2_training_history.json"
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)

    epochs = [h["epoch"] for h in history]
    losses = [h["loss"] for h in history]

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, losses, "b-", linewidth=2, label="Train")
    if "val_loss" in history[0]:
        val_losses = [h["val_loss"] for h in history]
        plt.plot(epochs, val_losses, "r-", linewidth=2, label="Val")
        plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("L1 Loss")
    plt.title("DDPM v2 Training Loss (FLAIR)")
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No training history found yet.")

## 14. Generate Synthetic FLAIR Images

In [ ]:
from src.med_ddpm_v2.sample import generate_synthetic

checkpoints = glob.glob(f"{CONFIG['checkpoint_dir']}/checkpoint_v2_epoch_*.pt")
if checkpoints:
    best = max(checkpoints, key=lambda p: int(Path(p).stem.split('_')[-1]))
    print(f"Generating from: {best}")
    generate_synthetic(
        checkpoint_path=best,
        output_dir=CONFIG["synthetic_dir"],
        config=CONFIG,
    )
else:
    print("No checkpoints found. Train the model first.")